# Header

In [ ]:
import xarray as xr
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cftime
import cartopy  # Map projections libary
import cartopy.crs as ccrs  # Projections list
import seaborn as sns
import cartopy.feature as cfeature
import time

# Structure of nudged experiments on Levante
<div class="alert alert-block alert-info">
<b> Information:</b>
We found a climate bias in the PD simulations and re-ran them. For this we had to change something in the production of the counterfactual SST forcing. In order for the simulations to be consistent, we also re-ran the NAT simulations.
</div>

All data is stored in \
`/work/bb1445/m300755/archive/` and\
`/work/bb1152/m300755/archive/`\
\
So far we have produced 4 experiments:
- Historical [HIST] (historical CO2) `CO2h_LUh_Aerh_YYYY`
- pre-industrial [NAT] (282ppm CO2) `sstClim_pi_YYYY`
- Historical without CO2 fertilization [THERM] `CO2hATMonly_LUh_Aerh_YYYY`
- Costant present-day forcing [PD] `sstClim_high_YYYY`

Because these original runs did not have daily global precipitation in the output (Only rainfall over land), we repeated the HIST and NAT simulations for ther period 1980 to 2024:
- `PREC_hist_YYYY`
- `PREC_pi_YYYY` (contains a sea ice related bug)

We could not run all experimens in one go and had to restart them, they are stored in different folders with the suffixes `_YYYY`. For example, you have to concatenate the pre-industrial simulation from `sstClim_pi_1940-2014`, and `sstClim_pi_2015-2024`.

### Here is a full list of the simulations:

| HIST (bb1445)     | NAT (bb1152) | THERM (bb1445)| PD (bb1152)|
| ------------- | ------------- | ------------- | ------------- |
| CO2h_LUh_Aerh_1940-2015 | sstClim_pi_1940-2014 | CO2hATMonly_LUh_Aerh_1940-2015 | sstClim_high_1940-2014 |
| CO2h_LUh_Aerh_1952-2015 | sstClim_pi_2015-2024 | CO2hATMonly_LUh_Aerh_1980-2015 | sstClim_high_2015-2024 |
| CO2h_LUh_Aerh_2015-2023 | sstClim_pi_2025_op | CO2hATMonly_LUh_Aerh_2015-2023 |
| CO2h_LUh_Aerh_2024 |  |
| sstClim_hist_2025_op | | | |

| PREC_hist (bb1445) | PREC_pi (bb1445) |
| ------------- | ------------- |
| PREC_hist_1980-2014 | PREC_pi_1980-2014 |
| PREC_hist_2015-2024 | PREC_pi_2015-2024 |

# List of output variables:

Within each of these folders, you can find the subfolders `/atm/hist` and `/lnd/hist` with most of the relevant output for the land and the atmoshere saven in streams h*i*.
You can find the variable name descriptions for the land and atmosphere [here](https://escomp.github.io/CTSM/release-clm5.0/users_guide/setting-up-and-running-a-case/master_list_file.html) and [here](https://web.archive.org/web/20250317194552/https://www2.cesm.ucar.edu/models/cesm2/atmosphere/docs/ug6/hist_flds_f2000.html).

Every simulation has an `lnd h0` stream with an extensive list of monthly variables.

### `lnd.h0` Monthly output

| Variable | Long name | Units |
|----------|------------|------------|
| mcsec | current seconds of current date | s |
| area | grid cell areas | km^2 |
| ACTUAL_IMMOB | actual N immobilization | gN/m^2/s |
| AGNPP | aboveground NPP | gC/m^2/s |
| ALT | current active layer thickness | m |
| ALTMAX | maximum annual active layer thickness | m |
| APPAR_TEMP | 2 m apparent temperature | C |
| APPAR_TEMP_R | Rural 2 m apparent temperature | C |
| APPAR_TEMP_U | Urban 2 m apparent temperature | C |
| AR | autotrophic respiration (MR + GR) | gC/m^2/s |
| ATM_TOPO | atmospheric surface height | m |
| BAF_CROP | fractional area burned for crop | s-1 |
| BAF_PEATF | fractional area burned in peatland | s-1 |
| BCDEP | total BC deposition (dry+wet) from atmosphere | kg/m^2/s |
| BGNPP | belowground NPP | gC/m^2/s |
| BTRAN2 | root zone soil wetness factor | unitless |
| BTRANMN | daily minimum of transpiration beta factor | unitless |
| CH4PROD | Gridcell total production of CH4 | gC/m2/s |
| CH4_SURF_AERE_SAT | aerenchyma surface CH4 flux for inundated area; (+ to atm) | mol/m2/s |
| CH4_SURF_AERE_UNSAT | aerenchyma surface CH4 flux for non-inundated area; (+ to atm) | mol/m2/s |
| CH4_SURF_DIFF_SAT | diffusive surface CH4 flux for inundated / lake area; (+ to atm) | mol/m2/s |
| CH4_SURF_DIFF_UNSAT | diffusive surface CH4 flux for non-inundated area; (+ to atm) | mol/m2/s |
| CH4_SURF_EBUL_SAT | ebullition surface CH4 flux for inundated / lake area; (+ to atm) | mol/m2/s |
| CH4_SURF_EBUL_UNSAT | ebullition surface CH4 flux for non-inundated area; (+ to atm) | mol/m2/s |
| COL_FIRE_CLOSS | total column-level fire C loss for non-peat fires outside land-type converted region | gC/m^2/s |
| COL_FIRE_NLOSS | total column-level fire N loss | gN/m^2/s |
| CONC_O2_SAT | O2 soil Concentration for inundated / lake area | mol/m3 |
| CONC_O2_UNSAT | O2 soil Concentration for non-inundated area | mol/m3 |
| COST_NACTIVE | Cost of active uptake | gN/gC |
| COST_NFIX | Cost of fixation | gN/gC |
| COST_NRETRANS | Cost of retranslocation | gN/gC |
| CPHASE | crop phenology phase | 0-not planted, 1-planted, 2-leaf emerge, 3-grain fill, 4-harvest |
| CPOOL | temporary photosynthate C pool | gC/m^2 |
| CROPPROD1C | 1-yr grain product C | gC/m^2 |
| CROPPROD1C_LOSS | loss from 1-yr grain product pool | gC/m^2/s |
| CROPPROD1N | 1-yr grain product N | gN/m^2 |
| CROPPROD1N_LOSS | loss from 1-yr grain product pool | gN/m^2/s |
| CROPSEEDC_DEFICIT | C used for crop seed that needs to be repaid | gC/m^2 |
| CWDC | CWD C | gC/m^2 |
| CWDC_LOSS | coarse woody debris C loss | gC/m^2/s |
| CWDC_vr | CWD C (vertically resolved) | gC/m^3 |
| CWDN | CWD N | gN/m^2 |
| CWDN_vr | CWD N (vertically resolved) | gN/m^3 |
| DEADCROOTC | dead coarse root C | gC/m^2 |
| DEADCROOTN | dead coarse root N | gN/m^2 |
| DEADSTEMC | dead stem C | gC/m^2 |
| DEADSTEMN | dead stem N | gN/m^2 |
| DENIT | total rate of denitrification | gN/m^2/s |
| DISCOI | 2 m Discomfort Index | C |
| DISCOIS | 2 m Stull Discomfort Index | C |
| DISCOIS_R | Rural 2 m Stull Discomfort Index | C |
| DISCOIS_U | Urban 2 m Stull Discomfort Index | C |
| DISCOI_R | Rural 2 m Discomfort Index | C |
| DISCOI_U | Urban 2 m Discomfort Index | C |
| DISPVEGC | displayed veg carbon, excluding storage and cpool | gC/m^2 |
| DISPVEGN | displayed vegetation nitrogen | gN/m^2 |
| DSL | dry surface layer thickness | mm |
| DSTDEP | total dust deposition (dry+wet) from atmosphere | kg/m^2/s |
| DSTFLXT | total surface dust emission | kg/m2/s |
| DWT_CONV_CFLUX | conversion C flux (immediate loss to atm) (0 at all times except first timestep of year) | gC/m^2/s |
| DWT_CONV_CFLUX_DRIBBLED | conversion C flux (immediate loss to atm), dribbled throughout the year | gC/m^2/s |
| DWT_CONV_NFLUX | conversion N flux (immediate loss to atm) (0 at all times except first timestep of year) | gN/m^2/s |
| DWT_CROPPROD1C_GAIN | landcover change-driven addition to 1-year crop product pool | gC/m^2/s |
| DWT_CROPPROD1N_GAIN | landcover change-driven addition to 1-year crop product pool | gN/m^2/s |
| DWT_SEEDN_TO_DEADSTEM | seed source to patch-level deadstem | gN/m^2/s |
| DWT_SEEDN_TO_LEAF | seed source to patch-level leaf | gN/m^2/s |
| DWT_SLASH_CFLUX | slash C flux to litter and CWD due to land use | gC/m^2/s |
| DWT_WOODPRODC_GAIN | landcover change-driven addition to wood product pools | gC/m^2/s |
| DWT_WOODPRODN_GAIN | landcover change-driven addition to wood product pools | gN/m^2/s |
| EFLXBUILD | building heat flux from change in interior building air temperature | W/m^2 |
| EFLX_DYNBAL | dynamic land cover change conversion energy flux | W/m^2 |
| EFLX_GRND_LAKE | net heat flux into lake/snow surface, excluding light transmission | W/m^2 |
| EFLX_LH_TOT | total latent heat flux [+ to atm] | W/m^2 |
| EFLX_LH_TOT_R | Rural total evaporation | W/m^2 |
| ELAI | exposed one-sided leaf area index | m^2/m^2 |
| EPT | 2 m Equiv Pot Temp | K |
| EPT_R | Rural 2 m Equiv Pot Temp | K |
| EPT_U | Urban 2 m Equiv Pot Temp | K |
| ER | total ecosystem respiration, autotrophic + heterotrophic | gC/m^2/s |
| ERRH2O | total water conservation error | mm |
| ERRH2OSNO | imbalance in snow depth (liquid water) | mm |
| ERRSEB | surface energy conservation error | W/m^2 |
| ERRSOI | soil/lake energy conservation error | W/m^2 |
| ERRSOL | solar radiation conservation error | W/m^2 |
| ESAI | exposed one-sided stem area index | m^2/m^2 |
| FAREA_BURNED | timestep fractional area burned | s-1 |
| FCEV | canopy evaporation | W/m^2 |
| FCH4 | Gridcell surface CH4 flux to atmosphere (+ to atm) | kgC/m2/s |
| FCH4TOCO2 | Gridcell oxidation of CH4 to CO2 | gC/m2/s |
| FCH4_DFSAT | CH4 additional flux due to changing fsat, vegetated landunits only | kgC/m2/s |
| FCOV | fractional impermeable area | unitless |
| FCTR | canopy transpiration | W/m^2 |
| FFIX_TO_SMINN | free living  N fixation to soil mineral N | gN/m^2/s |
| FGEV | ground evaporation | W/m^2 |
| FGR | heat flux into soil/snow including snow melt and lake / snow light transmission | W/m^2 |
| FGR12 | heat flux between soil layers 1 and 2 | W/m^2 |
| FH2OSFC | fraction of ground covered by surface water | unitless |
| FINUNDATED | fractional inundated area of vegetated columns | unitless |
| FIRA | net infrared (longwave) radiation | W/m^2 |
| FIRA_R | Rural net infrared (longwave) radiation | W/m^2 |
| FIRE | emitted infrared (longwave) radiation | W/m^2 |
| FIRE_R | Rural emitted infrared (longwave) radiation | W/m^2 |
| FLDS | atmospheric longwave radiation (downscaled to columns in glacier regions) | W/m^2 |
| FPI | fraction of potential immobilization | proportion |
| FPSN | photosynthesis | umol m-2 s-1 |
| FREE_RETRANSN_TO_NPOOL | deployment of retranslocated N | gN/m^2/s |
| FROOTC | fine root C | gC/m^2 |
| FROOTC_ALLOC | fine root C allocation | gC/m^2/s |
| FROOTC_LOSS | fine root C loss | gC/m^2/s |
| FROOTN | fine root N | gN/m^2 |
| FSA | absorbed solar radiation | W/m^2 |
| FSAT | fractional area with water table at surface | unitless |
| FSDS | atmospheric incident solar radiation | W/m^2 |
| FSDSND | direct nir incident solar radiation | W/m^2 |
| FSDSNDLN | direct nir incident solar radiation at local noon | W/m^2 |
| FSDSNI | diffuse nir incident solar radiation | W/m^2 |
| FSDSVD | direct vis incident solar radiation | W/m^2 |
| FSDSVDLN | direct vis incident solar radiation at local noon | W/m^2 |
| FSDSVI | diffuse vis incident solar radiation | W/m^2 |
| FSDSVILN | diffuse vis incident solar radiation at local noon | W/m^2 |
| FSH | sensible heat not including correction for land use change and rain/snow conversion | W/m^2 |
| FSH_G | sensible heat from ground | W/m^2 |
| FSH_PRECIP_CONVERSION | Sensible heat flux from conversion of rain/snow atm forcing | W/m^2 |
| FSH_R | Rural sensible heat | W/m^2 |
| FSH_RUNOFF_ICE_TO_LIQ | sensible heat flux generated from conversion of ice runoff to liquid | W/m^2 |
| FSH_TO_COUPLER | sensible heat sent to coupler (includes corrections for land use change, rain/snow conversion and conversion of ice runoff to liquid) | W/m^2 |
| FSH_V | sensible heat from veg | W/m^2 |
| FSM | snow melt heat flux | W/m^2 |
| FSNO | fraction of ground covered by snow | unitless |
| FSNO_EFF | effective fraction of ground covered by snow | unitless |
| FSR | reflected solar radiation | W/m^2 |
| FSRND | direct nir reflected solar radiation | W/m^2 |
| FSRNDLN | direct nir reflected solar radiation at local noon | W/m^2 |
| FSRNI | diffuse nir reflected solar radiation | W/m^2 |
| FSRVD | direct vis reflected solar radiation | W/m^2 |
| FSRVDLN | direct vis reflected solar radiation at local noon | W/m^2 |
| FSRVI | diffuse vis reflected solar radiation | W/m^2 |
| FUELC | fuel load | gC/m^2 |
| F_DENIT | denitrification flux | gN/m^2/s |
| F_N2O_DENIT | denitrification N2O flux | gN/m^2/s |
| F_N2O_NIT | nitrification N2O flux | gN/m^2/s |
| F_NIT | nitrification flux | gN/m^2/s |
| GPP | gross primary production | gC/m^2/s |
| GR | total growth respiration | gC/m^2/s |
| GRAINC | grain C (does not equal yield) | gC/m^2 |
| GRAINC_TO_FOOD | grain C to food | gC/m^2/s |
| GRAINC_TO_SEED | grain C to seed | gC/m^2/s |
| GRAINN | grain N | gN/m^2 |
| GROSS_NMIN | gross rate of N mineralization | gN/m^2/s |
| GSSHA | shaded leaf stomatal conductance | umol H20/m2/s |
| GSSHALN | shaded leaf stomatal conductance at local noon | umol H20/m2/s |
| GSSUN | sunlit leaf stomatal conductance | umol H20/m2/s |
| GSSUNLN | sunlit leaf stomatal conductance at local noon | umol H20/m2/s |
| H2OCAN | intercepted water | mm |
| H2OSFC | surface water depth | mm |
| H2OSNO | snow depth (liquid water) | mm |
| H2OSNO_TOP | mass of snow in top snow layer | kg/m2 |
| H2OSOI | volumetric soil water (vegetated landunits only) | mm3/mm3 |
| HEAT_CONTENT1 | initial gridcell total heat content | J/m^2 |
| HEAT_FROM_AC | sensible heat flux put into canyon due to heat removed from air conditioning | W/m^2 |
| HIA | 2 m NWS Heat Index | C |
| HIA_R | Rural 2 m NWS Heat Index | C |
| HIA_U | Urban 2 m NWS Heat Index | C |
| HR | total heterotrophic respiration | gC/m^2/s |
| HR_vr | total vertically resolved heterotrophic respiration | gC/m^3/s |
| HTOP | canopy top | m |
| HUMIDEX | 2 m Humidex | C |
| HUMIDEX_R | Rural 2 m Humidex | C |
| HUMIDEX_U | Urban 2 m Humidex | C |
| ICE_CONTENT1 | initial gridcell total ice content | mm |
| JMX25T | canopy profile of jmax | umol/m2/s |
| Jmx25Z | canopy profile of  vcmax25 predicted by LUNA model | umol/m2/s |
| LAISHA | shaded projected leaf area index | m^2/m^2 |
| LAISUN | sunlit projected leaf area index | m^2/m^2 |
| LAKEICEFRAC_SURF | surface lake layer ice mass fraction | unitless |
| LAKEICETHICK | thickness of lake ice (including physical expansion on freezing) | m |
| LAND_USE_FLUX | total C emitted from land cover conversion (smoothed over the year) and wood and grain product pools (NOTE: not a net value) | gC/m^2/s |
| LEAFC | leaf C | gC/m^2 |
| LEAFCN | Leaf CN ratio used for flexible CN | gC/gN |
| LEAFC_ALLOC | leaf C allocation | gC/m^2/s |
| LEAFC_CHANGE | C change in leaf | gC/m^2/s |
| LEAFC_LOSS | leaf C loss | gC/m^2/s |
| LEAFC_TO_LITTER_FUN | leaf C litterfall used by FUN | gC/m^2/s |
| LEAFN | leaf N | gN/m^2 |
| LEAFN_TO_LITTER | leaf N litterfall | gN/m^2/s |
| LEAF_MR | leaf maintenance respiration | gC/m^2/s |
| LFC2 | conversion area fraction of BET and BDT that burned | per sec |
| LIQCAN | intercepted liquid water | mm |
| LIQUID_CONTENT1 | initial gridcell total liq content | mm |
| LITFALL | litterfall (leaves and fine roots) | gC/m^2/s |
| LITR1C | LITR1 C | gC/m^2 |
| LITR1C_vr | LITR1 C (vertically resolved) | gC/m^3 |
| LITR1N | LITR1 N | gN/m^2 |
| LITR1N_vr | LITR1 N (vertically resolved) | gN/m^3 |
| LITR2C | LITR2 C | gC/m^2 |
| LITR2C_vr | LITR2 C (vertically resolved) | gC/m^3 |
| LITR2N | LITR2 N | gN/m^2 |
| LITR2N_vr | LITR2 N (vertically resolved) | gN/m^3 |
| LITR3C | LITR3 C | gC/m^2 |
| LITR3C_vr | LITR3 C (vertically resolved) | gC/m^3 |
| LITR3N | LITR3 N | gN/m^2 |
| LITR3N_vr | LITR3 N (vertically resolved) | gN/m^3 |
| LITTERC_HR | litter C heterotrophic respiration | gC/m^2/s |
| LITTERC_LOSS | litter C loss | gC/m^2/s |
| LIVECROOTC | live coarse root C | gC/m^2 |
| LIVECROOTN | live coarse root N | gN/m^2 |
| LIVESTEMC | live stem C | gC/m^2 |
| LIVESTEMN | live stem N | gN/m^2 |
| LNC | leaf N concentration | gN leaf/m^2 |
| MR | maintenance respiration | gC/m^2/s |
| NACTIVE | Mycorrhizal N uptake flux | gN/m^2/s |
| NACTIVE_NH4 | Mycorrhizal N uptake flux | gN/m^2/s |
| NACTIVE_NO3 | Mycorrhizal N uptake flux | gN/m^2/s |
| NAM | AM-associated N uptake flux | gN/m^2/s |
| NAM_NH4 | AM-associated N uptake flux | gN/m^2/s |
| NAM_NO3 | AM-associated N uptake flux | gN/m^2/s |
| NBP | net biome production, includes fire, landuse, harvest and hrv_xsmrpool flux (latter smoothed over the year), positive for sink (same as net carbon exchange between land and atmosphere) | gC/m^2/s |
| NDEPLOY | total N deployed in new growth | gN/m^2/s |
| NDEP_TO_SMINN | atmospheric N deposition to soil mineral N | gN/m^2/s |
| NECM | ECM-associated N uptake flux | gN/m^2/s |
| NECM_NH4 | ECM-associated N uptake flux | gN/m^2/s |
| NECM_NO3 | ECM-associated N uptake flux | gN/m^2/s |
| NEE | net ecosystem exchange of carbon, includes fire and hrv_xsmrpool (latter smoothed over the year), excludes landuse and harvest flux, positive for source | gC/m^2/s |
| NEM | Gridcell net adjustment to net carbon exchange passed to atm. for methane production | gC/m2/s |
| NEP | net ecosystem production, excludes fire, landuse, and harvest flux, positive for sink | gC/m^2/s |
| NET_NMIN | net rate of N mineralization | gN/m^2/s |
| NFERTILIZATION | fertilizer added | gN/m^2/s |
| NFIRE | fire counts valid only in Reg.C | counts/km2/sec |
| NFIX | Symbiotic BNF uptake flux | gN/m^2/s |
| NNONMYC | Non-mycorrhizal N uptake flux | gN/m^2/s |
| NNONMYC_NH4 | Non-mycorrhizal N uptake flux | gN/m^2/s |
| NNONMYC_NO3 | Non-mycorrhizal N uptake flux | gN/m^2/s |
| NPASSIVE | Passive N uptake flux | gN/m^2/s |
| NPOOL | temporary plant N pool | gN/m^2 |
| NPP | net primary production | gC/m^2/s |
| NPP_GROWTH | Total C used for growth in FUN | gC/m^2/s |
| NPP_NACTIVE | Mycorrhizal N uptake used C | gC/m^2/s |
| NPP_NACTIVE_NH4 | Mycorrhizal N uptake use C | gC/m^2/s |
| NPP_NACTIVE_NO3 | Mycorrhizal N uptake used C | gC/m^2/s |
| NPP_NAM | AM-associated N uptake used C | gC/m^2/s |
| NPP_NAM_NH4 | AM-associated N uptake use C | gC/m^2/s |
| NPP_NAM_NO3 | AM-associated N uptake use C | gC/m^2/s |
| NPP_NECM | ECM-associated N uptake used C | gC/m^2/s |
| NPP_NECM_NH4 | ECM-associated N uptake use C | gC/m^2/s |
| NPP_NECM_NO3 | ECM-associated N uptake used C | gC/m^2/s |
| NPP_NFIX | Symbiotic BNF uptake used C | gC/m^2/s |
| NPP_NNONMYC | Non-mycorrhizal N uptake used C | gC/m^2/s |
| NPP_NNONMYC_NH4 | Non-mycorrhizal N uptake use C | gC/m^2/s |
| NPP_NNONMYC_NO3 | Non-mycorrhizal N uptake use C | gC/m^2/s |
| NPP_NRETRANS | Retranslocated N uptake flux | gC/m^2/s |
| NPP_NUPTAKE | Total C used by N uptake in FUN | gC/m^2/s |
| NRETRANS | Retranslocated N uptake flux | gN/m^2/s |
| NRETRANS_REG | Retranslocated N uptake flux | gN/m^2/s |
| NRETRANS_SEASON | Retranslocated N uptake flux | gN/m^2/s |
| NRETRANS_STRESS | Retranslocated N uptake flux | gN/m^2/s |
| NUPTAKE | Total N uptake of FUN | gN/m^2/s |
| NUPTAKE_NPP_FRACTION | frac of NPP used in N uptake | - |
| OCDEP | total OC deposition (dry+wet) from atmosphere | kg/m^2/s |
| O_SCALAR | fraction by which decomposition is reduced due to anoxia | unitless |
| PARVEGLN | absorbed par by vegetation at local noon | W/m^2 |
| PBOT | atmospheric pressure at surface (downscaled to columns in glacier regions) | Pa |
| PCH4 | atmospheric partial pressure of CH4 | Pa |
| PCO2 | atmospheric partial pressure of CO2 | Pa |
| PCT_CFT | % of each crop on the crop landunit | % |
| PCT_GLC_MEC | % of each GLC elevation class on the glc_mec landunit | % |
| PCT_LANDUNIT | % of each landunit on grid cell | % |
| PCT_NAT_PFT | % of each PFT on the natural vegetation (i.e., soil) landunit | % |
| PFT_FIRE_CLOSS | total patch-level fire C loss for non-peat fires outside land-type converted region | gC/m^2/s |
| PFT_FIRE_NLOSS | total patch-level fire N loss | gN/m^2/s |
| PLANT_NDEMAND | N flux required to support initial GPP | gN/m^2/s |
| POTENTIAL_IMMOB | potential N immobilization | gN/m^2/s |
| POT_F_DENIT | potential denitrification flux | gN/m^2/s |
| POT_F_NIT | potential nitrification flux | gN/m^2/s |
| PSNSHA | shaded leaf photosynthesis | umolCO2/m^2/s |
| PSNSHADE_TO_CPOOL | C fixation from shaded canopy | gC/m^2/s |
| PSNSUN | sunlit leaf photosynthesis | umolCO2/m^2/s |
| PSNSUN_TO_CPOOL | C fixation from sunlit canopy | gC/m^2/s |
| Q2M | 2m specific humidity | kg/kg |
| QBOT | atmospheric specific humidity (downscaled to columns in glacier regions) | kg/kg |
| QCHARGE | aquifer recharge rate (vegetated landunits only) | mm/s |
| QDRAI | sub-surface drainage | mm/s |
| QDRAI_PERCH | perched wt drainage | mm/s |
| QDRAI_XS | saturation excess drainage | mm/s |
| QDRIP | throughfall | mm/s |
| QFLOOD | runoff from river flooding | mm/s |
| QFLX_DEW_GRND | ground surface dew formation | mm H2O/s |
| QFLX_DEW_SNOW | surface dew added to snow pacK | mm H2O/s |
| QFLX_EVAP_TOT | qflx_evap_soi + qflx_evap_can + qflx_tran_veg | kg m-2 s-1 |
| QFLX_ICE_DYNBAL | ice dynamic land cover change conversion runoff flux | mm/s |
| QFLX_LIQ_DYNBAL | liq dynamic land cover change conversion runoff flux | mm/s |
| QFLX_SNOW_DRAIN | drainage from snow pack | mm/s |
| QFLX_SNOW_DRAIN_ICE | drainage from snow pack melt (ice landunits only) | mm/s |
| QFLX_SUB_SNOW | sublimation rate from snow pack (also includes bare ice sublimation from glacier columns) | mm H2O/s |
| QH2OSFC | surface water runoff | mm/s |
| QICE | ice growth/melt | mm/s |
| QICE_FRZ | ice growth | mm/s |
| QICE_MELT | ice melt | mm/s |
| QINFL | infiltration | mm/s |
| QINTR | interception | mm/s |
| QIRRIG | water added through irrigation | mm/s |
| QOVER | surface runoff | mm/s |
| QRGWL | surface runoff at glaciers (liquid only), wetlands, lakes; also includes melted ice runoff from QSNWCPICE | mm/s |
| QRUNOFF | total liquid runoff not including correction for land use change | mm/s |
| QRUNOFF_ICE | total liquid runoff not incl corret for LULCC (ice landunits only) | mm/s |
| QRUNOFF_ICE_TO_COUPLER | total ice runoff sent to coupler (includes corrections for land use change) | mm/s |
| QRUNOFF_RAIN_TO_SNOW_CONVERSION | liquid runoff from rain-to-snow conversion when this conversion leads to immediate runoff | mm/s |
| QRUNOFF_TO_COUPLER | total liquid runoff sent to coupler (includes corrections for land use change) | mm/s |
| QSNOCPLIQ | excess liquid h2o due to snow capping not including correction for land use change | mm H2O/s |
| QSNOEVAP | evaporation from snow | mm/s |
| QSNOFRZ | column-integrated snow freezing rate | kg/m2/s |
| QSNOFRZ_ICE | column-integrated snow freezing rate (ice landunits only) | mm/s |
| QSNOMELT | snow melt rate | mm/s |
| QSNOMELT_ICE | snow melt (ice landunits only) | mm/s |
| QSNO_TEMPUNLOAD | canopy snow temp unloading | mm/s |
| QSNO_WINDUNLOAD | canopy snow wind unloading | mm/s |
| QSNWCPICE | excess solid h2o due to snow capping not including correction for land use change | mm H2O/s |
| QSOIL | Ground evaporation (soil/snow evaporation + soil/snow sublimation - dew) | mm/s |
| QSOIL_ICE | Ground evaporation (ice landunits only) | mm/s |
| QVEGE | canopy evaporation | mm/s |
| QVEGT | canopy transpiration | mm/s |
| RAIN | atmospheric rain, after rain/snow repartitioning based on temperature | mm/s |
| RAIN_FROM_ATM | atmospheric rain received from atmosphere (pre-repartitioning) | mm/s |
| RETRANSN | plant pool of retranslocated N | gN/m^2 |
| RETRANSN_TO_NPOOL | deployment of retranslocated N | gN/m^2/s |
| RH2M | 2m relative humidity | % |
| RR | root respiration (fine root MR + total root GR) | gC/m^2/s |
| RSSHA | shaded leaf stomatal resistance | s/m |
| RSSUN | sunlit leaf stomatal resistance | s/m |
| SABG | solar rad absorbed by ground | W/m^2 |
| SABG_PEN | Rural solar rad penetrating top soil or snow layer | watt/m^2 |
| SABV | solar rad absorbed by veg | W/m^2 |
| SEEDC | pool for seeding new PFTs via dynamic landcover | gC/m^2 |
| SEEDN | pool for seeding new PFTs via dynamic landcover | gN/m^2 |
| SLASH_HARVESTC | slash harvest carbon (to litter) | gC/m^2/s |
| SMINN | soil mineral N | gN/m^2 |
| SMINN_TO_NPOOL | deployment of soil mineral N uptake | gN/m^2/s |
| SMINN_TO_PLANT | plant uptake of soil mineral N | gN/m^2/s |
| SMINN_TO_PLANT_FUN | Total soil N uptake of FUN | gN/m^2/s |
| SMINN_vr | soil mineral N | gN/m^3 |
| SMIN_NH4 | soil mineral NH4 | gN/m^2 |
| SMIN_NH4_vr | soil mineral NH4 (vert. res.) | gN/m^3 |
| SMIN_NO3 | soil mineral NO3 | gN/m^2 |
| SMIN_NO3_LEACHED | soil NO3 pool loss to leaching | gN/m^2/s |
| SMIN_NO3_RUNOFF | soil NO3 pool loss to runoff | gN/m^2/s |
| SMIN_NO3_vr | soil mineral NO3 (vert. res.) | gN/m^3 |
| SMP | soil matric potential (vegetated landunits only) | mm |
| SNOBCMCL | mass of BC in snow column | kg/m2 |
| SNOBCMSL | mass of BC in top snow layer | kg/m2 |
| SNOCAN | intercepted snow | mm |
| SNODSTMCL | mass of dust in snow column | kg/m2 |
| SNODSTMSL | mass of dust in top snow layer | kg/m2 |
| SNOFSRND | direct nir reflected solar radiation from snow | W/m^2 |
| SNOFSRNI | diffuse nir reflected solar radiation from snow | W/m^2 |
| SNOFSRVD | direct vis reflected solar radiation from snow | W/m^2 |
| SNOFSRVI | diffuse vis reflected solar radiation from snow | W/m^2 |
| SNOINTABS | Fraction of incoming solar absorbed by lower snow layers | - |
| SNOOCMCL | mass of OC in snow column | kg/m2 |
| SNOOCMSL | mass of OC in top snow layer | kg/m2 |
| SNOTXMASS | snow temperature times layer mass, layer sum; to get mass-weighted temperature, divide by (SNOWICE+SNOWLIQ) | K kg/m2 |
| SNOUNLOAD | Canopy snow unloading | mm |
| SNOW | atmospheric snow, after rain/snow repartitioning based on temperature | mm/s |
| SNOWDP | gridcell mean snow height | m |
| SNOWICE | snow ice | kg/m2 |
| SNOWLIQ | snow liquid water | kg/m2 |
| SNOW_DEPTH | snow height of snow covered area | m |
| SNOW_FROM_ATM | atmospheric snow received from atmosphere (pre-repartitioning) | mm/s |
| SNOW_SINKS | snow sinks (liquid water) | mm/s |
| SNOW_SOURCES | snow sources (liquid water) | mm/s |
| SOIL1C | SOIL1 C | gC/m^2 |
| SOIL1C_vr | SOIL1 C (vertically resolved) | gC/m^3 |
| SOIL1N | SOIL1 N | gN/m^2 |
| SOIL1N_vr | SOIL1 N (vertically resolved) | gN/m^3 |
| SOIL2C | SOIL2 C | gC/m^2 |
| SOIL2C_vr | SOIL2 C (vertically resolved) | gC/m^3 |
| SOIL2N | SOIL2 N | gN/m^2 |
| SOIL2N_vr | SOIL2 N (vertically resolved) | gN/m^3 |
| SOIL3C | SOIL3 C | gC/m^2 |
| SOIL3C_vr | SOIL3 C (vertically resolved) | gC/m^3 |
| SOIL3N | SOIL3 N | gN/m^2 |
| SOIL3N_vr | SOIL3 N (vertically resolved) | gN/m^3 |
| SOILC_CHANGE | C change in soil | gC/m^2/s |
| SOILC_HR | soil C heterotrophic respiration | gC/m^2/s |
| SOILC_vr | SOIL C (vertically resolved) | gC/m^3 |
| SOILICE | soil ice (vegetated landunits only) | kg/m2 |
| SOILLIQ | soil liquid water (vegetated landunits only) | kg/m2 |
| SOILN_vr | SOIL N (vertically resolved) | gN/m^3 |
| SOILRESIS | soil resistance to evaporation | s/m |
| SOILWATER_10CM | soil liquid water + ice in top 10cm of soil (veg landunits only) | kg/m2 |
| SOMC_FIRE | C loss due to peat burning | gC/m^2/s |
| SOM_C_LEACHED | total flux of C from SOM pools due to leaching | gC/m^2/s |
| SR | total soil respiration (HR + root resp) | gC/m^2/s |
| STORVEGC | stored vegetation carbon, excluding cpool | gC/m^2 |
| STORVEGN | stored vegetation nitrogen | gN/m^2 |
| SUPPLEMENT_TO_SMINN | supplemental N supply | gN/m^2/s |
| SWBGT | 2 m Simplified Wetbulb Globe Temp | C |
| SWBGT_R | Rural 2 m Simplified Wetbulb Globe Temp | C |
| SWBGT_U | Urban 2 m Simplified Wetbulb Globe Temp | C |
| SWMP65 | 2 m Swamp Cooler Temp 65% Eff | C |
| SWMP65_R | Rural 2 m Swamp Cooler Temp 65% Eff | C |
| SWMP65_U | Urban 2 m Swamp Cooler Temp 65% Eff | C |
| SWMP80 | 2 m Swamp Cooler Temp 80% Eff | C |
| SWMP80_R | Rural 2 m Swamp Cooler Temp 80% Eff | C |
| SWMP80_U | Urban 2 m Swamp Cooler Temp 80% Eff | C |
| TAUX | zonal surface stress | kg/m/s^2 |
| TAUY | meridional surface stress | kg/m/s^2 |
| TBOT | atmospheric air temperature (downscaled to columns in glacier regions) | K |
| TBUILD | internal urban building air temperature | K |
| TEQ | 2 m Equiv Temp | K |
| TEQ_R | Rural 2 m Equiv Temp | K |
| TEQ_U | Urban 2 m Equiv Temp | K |
| TG | ground temperature | K |
| TH2OSFC | surface water temperature | K |
| THBOT | atmospheric air potential temperature (downscaled to columns in glacier regions) | K |
| THIC | 2 m Temp Hum Index Comfort | C |
| THIC_R | Rural 2 m Temp Hum Index Comfort | C |
| THIC_U | Urban 2 m Temp Hum Index Comfort | C |
| THIP | 2 m Temp Hum Index Physiology | C |
| THIP_R | Rural 2 m Temp Hum Index Physiology | C |
| THIP_U | Urban 2 m Temp Hum Index Physiology | C |
| TKE1 | top lake level eddy thermal conductivity | W/(mK) |
| TLAI | total projected leaf area index | m^2/m^2 |
| TLAKE | lake temperature | K |
| TOTCOLC | total column carbon, incl veg and cpool but excl product pools | gC/m^2 |
| TOTCOLCH4 | total belowground CH4 (0 for non-lake special landunits in the absence of dynamic landunits) | gC/m2 |
| TOTCOLN | total column-level N, excluding product pools | gN/m^2 |
| TOTECOSYSC | total ecosystem carbon, incl veg but excl cpool and product pools | gC/m^2 |
| TOTECOSYSN | total ecosystem N, excluding product pools | gN/m^2 |
| TOTLITC | total litter carbon | gC/m^2 |
| TOTLITC_1m | total litter carbon to 1 meter depth | gC/m^2 |
| TOTLITN | total litter N | gN/m^2 |
| TOTLITN_1m | total litter N to 1 meter | gN/m^2 |
| TOTPFTC | total patch-level carbon, including cpool | gC/m^2 |
| TOTPFTN | total patch-level nitrogen | gN/m^2 |
| TOTSOILICE | vertically summed soil cie (veg landunits only) | kg/m2 |
| TOTSOILLIQ | vertically summed soil liquid water (veg landunits only) | kg/m2 |
| TOTSOMC | total soil organic matter carbon | gC/m^2 |
| TOTSOMC_1m | total soil organic matter carbon to 1 meter depth | gC/m^2 |
| TOTSOMN | total soil organic matter N | gN/m^2 |
| TOTSOMN_1m | total soil organic matter N to 1 meter | gN/m^2 |
| TOTVEGC | total vegetation carbon, excluding cpool | gC/m^2 |
| TOTVEGN | total vegetation nitrogen | gN/m^2 |
| TOT_WOODPRODC | total wood product C | gC/m^2 |
| TOT_WOODPRODC_LOSS | total loss from wood product pools | gC/m^2/s |
| TOT_WOODPRODN | total wood product N | gN/m^2 |
| TOT_WOODPRODN_LOSS | total loss from wood product pools | gN/m^2/s |
| TPU25T | canopy profile of tpu | umol/m2/s |
| TREFMNAV | daily minimum of average 2-m temperature | K |
| TREFMXAV | daily maximum of average 2-m temperature | K |
| TSA | 2m air temperature | K |
| TSAI | total projected stem area index | m^2/m^2 |
| TSKIN | skin temperature | K |
| TSL | temperature of near-surface soil layer (vegetated landunits only) | K |
| TSOI | soil temperature (vegetated landunits only) | K |
| TSOI_10CM | soil temperature in top 10cm of soil | K |
| TSOI_ICE | soil temperature (ice landunits only) | K |
| TV | vegetation temperature | K |
| TWS | total water storage | mm |
| T_SCALAR | temperature inhibition of decomposition | unitless |
| U10 | 10-m wind | m/s |
| U10_DUST | 10-m wind for dust model | m/s |
| URBAN_AC | urban air conditioning flux | W/m^2 |
| URBAN_HEAT | urban heating flux | W/m^2 |
| VCMX25T | canopy profile of vcmax25 | umol/m2/s |
| VEGWP | vegetation water matric potential for sun/sha canopy,xyl,root segments | mm |
| VOLR | river channel total water storage | m3 |
| VOLRMCH | river channel main channel water storage | m3 |
| Vcmx25Z | canopy profile of vcmax25 predicted by LUNA model | umol/m2/s |
| WA | water in the unconfined aquifer (vegetated landunits only) | mm |
| WASTEHEAT | sensible heat flux from heating/cooling sources of urban waste heat | W/m^2 |
| WBA | 2 m Wet Bulb | C |
| WBA_R | Rural 2 m Wet Bulb | C |
| WBA_U | Urban 2 m Wet Bulb | C |
| WBT | 2 m Stull Wet Bulb | C |
| WBT_R | Rural 2 m Stull Wet Bulb | C |
| WBT_U | Urban 2 m Stull Wet Bulb | C |
| WIND | atmospheric wind velocity magnitude | m/s |
| WOODC | wood C | gC/m^2 |
| WOODC_ALLOC | wood C eallocation | gC/m^2/s |
| WOODC_LOSS | wood C loss | gC/m^2/s |
| WOOD_HARVESTC | wood harvest carbon (to product pools) | gC/m^2/s |
| WOOD_HARVESTN | wood harvest N (to product pools) | gN/m^2/s |
| WTGQ | surface tracer conductance | m/s |
| W_SCALAR | Moisture (dryness) inhibition of decomposition | unitless |
| XSMRPOOL | temporary photosynthate C pool | gC/m^2 |
| XSMRPOOL_RECOVER | C flux assigned to recovery of negative xsmrpool | gC/m^2/s |
| ZBOT | atmospheric reference height | m |
| ZWT | water table depth (vegetated landunits only) | m |
| ZWT_CH4_UNSAT | depth of water table for methane production used in non-inundated area | m |
| ZWT_PERCH | perched water table depth (vegetated landunits only) | m |

## Output of the 1st batch of simulaions (CO2x)

### `lnd.h1` Daily carbon related variables

| Variable | Long name | Units |
|----------|------------|------------|
| AR | autotrophic respiration (MR + GR) | gC/m^2/s |
| FAREA_BURNED | timestep fractional area burned | s-1 |
| GPP | gross primary production | gC/m^2/s |
| HR | total heterotrophic respiration | gC/m^2/s |
| LAND_USE_FLUX | total C emitted from land cover conversion (smoothed over the year) and wood and grain product pools (NOTE: not a net value) | gC/m^2/s |
| NBP | net biome production, includes fire, landuse, harvest and hrv_xsmrpool flux (latter smoothed over the year), positive for sink (same as net carbon exchange between land and atmosphere) | gC/m^2/s |
| SMIN_NO3_LEACHED | soil NO3 pool loss to leaching | gN/m^2/s |
| TLAI | total projected leaf area index | m^2/m^2 |
| TOTFIRE | total ecosystem fire losses | gC/m^2/s |

### `lnd.h2` Daily climate variables

| Variable | Long name | Units |
|----------|------------|------------|
| EFLX_LH_TOT | total latent heat flux [+ to atm] | W/m^2 |
| FSDS | atmospheric incident solar radiation | W/m^2 |
| FSH | sensible heat not including correction for land use change and rain/snow conversion | W/m^2 |
| FSNO | fraction of ground covered by snow | unitless |
| PBOT | atmospheric pressure at surface (downscaled to columns in glacier regions) | Pa |
| QFLX_EVAP_TOT | qflx_evap_soi + qflx_evap_can + qflx_tran_veg | kg m-2 s-1 |
| QRUNOFF | total liquid runoff not including correction for land use change | mm/s |
| QVEGT | canopy transpiration | mm/s |
| RAIN | atmospheric rain, after rain/snow repartitioning based on temperature | mm/s |
| RH2M | 2m relative humidity | % |
| RH2M_U | Urban 2m relative humidity | % |
| SOILWATER_10CM | soil liquid water + ice in top 10cm of soil (veg landunits only) | kg/m2 |
| TOTSOILLIQ | vertically summed soil liquid water (veg landunits only) | kg/m2 |
| TREFMNAV | daily minimum of average 2-m temperature | K |
| TREFMXAV | daily maximum of average 2-m temperature | K |
| TREFMXAV_U | Urban daily maximum of average 2-m temperature | K |
| TSA | 2m air temperature | K |
| TSA_U | Urban 2m air temperature | K |
| TSOI_10CM | soil temperature in top 10cm of soil | K |
| U10 | 10-m wind | m/s |

### `lnd.h3` Daily heat-stress indices

| Variable | Long name | Units |
|----------|------------|------------|
| DISCOI | 2 m Discomfort Index | C |
| DISCOI_U | Urban 2 m Discomfort Index | C |
| H2OSOI | volumetric soil water (vegetated landunits only) | mm3/mm3 |
| HIA | 2 m NWS Heat Index | C |
| HUMIDEX | 2 m Humidex | C |
| HUMIDEX_R | Rural 2 m Humidex | C |
| HUMIDEX_U | Urban 2 m Humidex | C |
| QRUNOFF_U | Urban total runoff | mm/s |
| THIP | 2 m Temp Hum Index Physiology | C |
| WBA | 2 m Wet Bulb | C |
| WBA_U | Urban 2 m Wet Bulb | C |

### `lnd.h4` Maximum daily heat-stress indices

| Variable | Long name | Units |
|----------|------------|------------|
| DISCOI | 2 m Discomfort Index | C |
| WBA | 2 m Wet Bulb | C |
| WBA_U | Urban 2 m Wet Bulb | C |

### `lnd.h5` Monthly output of the crop model and use of airconditioning

| Variable | Long name | Units |
|----------|------------|------------|
| ALBD | surface albedo (direct) | proportion |
| CPHASE | crop phenology phase | 0-not planted, 1-planted, 2-leaf emerge, 3-grain fill, 4-harvest |
| CROPPROD1C | 1-yr grain product C | gC/m^2 |
| GDD8 | Growing degree days base  8C from planting | ddays |
| GDDPLANT | Accumula`ted growing degree days past planting date for crop | ddays |
| GRAINC_TO_FOOD | grain C to food | gC/m^2/s |
| PCO2 | atmospheric partial pressure of CO2 | Pa |
| QIRRIG_DEMAND | irrigation demand | mm/s |
| URBAN_AC | urban air conditioning flux | W/m^2 |

### `lnd.h6` Monthly GPP and crop yield output at the PFT level

| Variable | Long name | Units |
|----------|------------|------------|
| GPP | gross primary production | gC/m^2/s |
| GRAINC_TO_FOOD | grain C to food | gC/m^2/s |
| NBP | net biome production, includes fire, landuse, harvest and hrv_xsmrpool flux (latter smoothed over the year), positive for sink (same as net carbon exchange between land and atmosphere) | gC/m^2/s |

### `atm.h1` Monthly output

| Variable | Long name | Units |
|----------|------------|------------|
| PRECC | Convective precipitation rate (liq + ice) | m/s |
| PRECL | Large-scale (stable) precipitation rate (liq + ice) | m/s |
| PSL | Sea level pressure | Pa |
| QREFHT | Reference height humidity | kg/kg |
| RHREFHT | Reference height relative humidity | fraction |
| SST | sea surface temperature | K |
| TREFHT | Reference height temperature | K |
| TREFHTMN | Minimum reference height temperature over output period | K |
| TREFHTMX | Maximum reference height temperature over output period | K |
| TREFMNAV | Average of TREFHT daily minimum | K |
| TREFMXAV | Average of TREFHT daily maximum | K |
| Z500 | Geopotential Z at 500 mbar pressure surface | m |

### `atm.h2` Monthly output

| Variable | Long name | Units |
|----------|------------|------------|
| P0 | reference pressure | Pa |
| mdt | timestep | s |
| sol_tsi | total solar irradiance | W/m2 |
| AEROD_v | Total Aerosol Optical Depth in visible band | 1 |
| FLDS | Downwelling longwave flux at surface | W/m2 |
| FLNS | Net longwave flux at surface | W/m2 |
| FSDS | Downwelling solar flux at surface | W/m2 |
| FSNS | Net solar flux at surface | W/m2 |
| FSNTOA | Net solar flux at top of atmosphere | W/m2 |
| LHFLX | Surface latent heat flux | W/m2 |
| SHFLX | Surface sensible heat flux | W/m2 |
| cb_CO2_c | CO2 column burden used in climate calculation | kg/m^2 |
| so4_a1DDF | so4_a1 dry deposition flux at bottom (grav + turb) | kg/m2/s |

## Output of the 2nd batch of simulaions (`PREC_x` and `sstClim`)

### `lnd` is a subset of the above

### `atm.h1` Daily output

| Variable | Long name | Units |
|----------|------------|------------|
| CAPE | Convectively available potential energy | J/kg |
| CIN | Convective inhibition | J/kg |
| FREQSH | Fractional occurance of shallow convection | fraction |
| FREQZM | Fractional occurance of ZM convection | fraction |
| OMEGA500 | Vertical velocity at 500 mbar pressure surface | Pa/s |
| OMEGA850 | Vertical velocity at 850 mbar pressure surface | Pa/s |
| PBLH | PBL height | m |
| PCONVB | convection base pressure | Pa |
| PCONVT | convection top  pressure | Pa |
| PRECC | Convective precipitation rate (liq + ice) | m/s |
| PRECL | Large-scale (stable) precipitation rate (liq + ice) | m/s |
| PRECSC | Convective snow rate (water equivalent) | m/s |
| PRECSL | Large-scale (stable) snow rate (water equivalent) | m/s |
| PRECT | Total (convective and large-scale) precipitation rate (liq + ice) | m/s |
| PSL | Sea level pressure | Pa |
| RHREFHT | Reference height relative humidity | fraction |
| TMQ | Total (vertically integrated) precipitable water | kg/m2 |
| TREFHT | Reference height temperature | K |
| Z500 | Geopotential Z at 500 mbar pressure surface | m |

### `atm.h2` Daily radiation

| Variable | Long name | Units |
|----------|------------|------------|
| FLDS | Downwelling longwave flux at surface | W/m2 |
| FLNS | Net longwave flux at surface | W/m2 |
| FSNS | Net solar flux at surface | W/m2 |
| FSNTOA | Net solar flux at top of atmosphere | W/m2 |
| LHFLX | Surface latent heat flux | W/m2 |
| SHFLX | Surface sensible heat flux | W/m2 |

### `rof.h1` Daily river discharge

| Variable | Long name | Units |
|----------|------------|------------|
| areatotal | basin upstream areatotal | m2 |
| areatotal2 | computed basin upstream areatotal | m2 |
| DIRECT_DISCHARGE_TO_OCEAN_ICE | MOSART direct discharge into ocean: ICE | m3/s |
| DIRECT_DISCHARGE_TO_OCEAN_LIQ | MOSART direct discharge into ocean: LIQ | m3/s |
| RIVER_DISCHARGE_OVER_LAND_ICE | MOSART river basin flow: ICE | m3/s |
| RIVER_DISCHARGE_OVER_LAND_LIQ | MOSART river basin flow: LIQ | m3/s |
| TOTAL_DISCHARGE_TO_OCEAN_ICE | MOSART total discharge into ocean: ICE | m3/s |
| TOTAL_DISCHARGE_TO_OCEAN_LIQ | MOSART total discharge into ocean: LIQ | m3/s |

# Some things to consider when loading the data

## Load data from `lnd`

There are two things to consider when loading daily land data. 
- CESM2 manages time steps different to other common packages such as xarray. For example, if you load daily `lnd` data (which is stored at 12:00 in CESM2) it will turn into the following day by xarray. I use `ncview` to check the original time steps (This also happens with onthly data, where January 15 is turned into February).
- The first time step of a newly initialized simulation is not the 1st day daily average, but instread the initial condition. Because every simulation is re-initialized in 2015, you have to discard 2015-01-01.

In [ ]:
# You can get a list of available daily climate in h2 like this:
meta = xr.open_dataset('/work/bb1445/m300755/archive/CO2pi_LUh_Aerpi_1940-2015/lnd/hist/CO2pi_LUh_Aerpi_1940-2015.clm2.h2.1940-01-02-00000.nc')
meta

In [ ]:
# list of variables to load:
var_name=['TSA', 'TREFMXAV', 'FSDS']

# Load three folders of NAT siumlation
pi1_f = sorted(glob.glob('/work/bb1445/m300755/archive/CO2pi_LUh_Aerpi_1940-2015/lnd/hist/CO2pi_LUh_Aerpi_1940-2015.clm2.h2.*'))
pi1 = xr.open_mfdataset(pi1_f, concat_dim='time', combine='nested')[var_name]

pi2_f = sorted(glob.glob('/work/bb1445/m300755/archive/CO2pi_LUh_Aerpi_1952-2015/lnd/hist/CO2pi_LUh_Aerpi_1952-2015.clm2.h2.*'))
pi2 = xr.open_mfdataset(pi2_f, concat_dim='time', combine='nested')[var_name]

pi3_f = sorted(glob.glob('/work/bb1445/m300755/archive/CO2pi_LUh_Aerpi_2015-2023/lnd/hist/CO2pi_LUh_Aerpi_2015-2023.clm2.h2.*'))
pi3 = xr.open_mfdataset(pi3_f, concat_dim='time', combine='nested')[var_name]


pi = xr.merge([pi1,
               pi2, 
               pi3.isel(time=slice(1, None))])  # Excluding the initial condition in the 2015 restart

pi = pi['time'] - pd.Timedelta(days=1)